In [ ]:
### Install requirements
from pathlib import Path

REPOSITORY = "https://github.com/lmingari/olot-course.git"
PROJECT_FILE = "pyproject.toml"
BRANCH = "extra"

if not Path(PROJECT_FILE).exists():
    !git clone -q -b {BRANCH} {REPOSITORY}
    %cd olot-course
    !pip install -q -e ".[data,ml]"

# 4. Generative AI
---

## Problem statement

Let $\{x_i\}_{i=1}^{N}$ be a dataset of independent observations assumed to be drawn from an unknown probability distribution:

$$x_i \sim p_{\text{data}}, \qquad i = 1, \dots, N.$$

The objective of generative modeling is to produce **new samples** $x \sim p_{\text{data}}$, that is, new fields that are statistically indistinguishable from those generated by the original simulations.

## Why is this difficult?

The distribution $p_{\text{data}}$ is **intractable**:

- It is not available in closed form, so the density $p_{\text{data}}(x)$ cannot be evaluated
- It can only be accessed through a finite set of samples, so there is no direct procedure to draw new ones
- The data (images, simulation fields, etc.) are complex and highly structured, making it difficult, or even impossible, to estimate the density or sample from it directly

## Approach

Instead of sampling from $p_{\text{data}}$ directly, we introduce a reference distribution $p_{\text{simple}}$ from which sampling is straightforward, for example a standard Gaussian:

$$z \sim p_{\text{simple}} = \mathcal{N}(0, I).$$

The goal is to learn a transformation that maps samples of $p_{\text{simple}}$ to samples of $p_{\text{data}}$:

$$z \sim p_{\text{simple}} \;\longmapsto\; x \sim p_{\text{data}}.$$

## Generation as a dynamical system

The transformation can be described as a **flow in time**. A trajectory $x_t$ moves in a **wind field** $u_\theta(x, t)$ for $t \in [0, 1]$, so that:

* it starts from a noise sample, $x_0 = z \sim p_{\text{simple}}$, and
* it ends at a data sample, $x_1 \sim p_{\text{data}}$.

**Flow models (ODE).** The particle follows the wind deterministically:

$$\frac{dx_t}{dt} = u_\theta(x_t, t), \qquad x_0 = z \sim p_{\text{simple}}.$$

**Diffusion models (SDE).** The particle follows the wind but is also shaken by random noise, with $W_t$ a Brownian motion and $\sigma_t$ the noise strength:

$$dx_t = u_\theta(x_t, t)\,dt + \sigma_t\,dW_t, \qquad x_0 = z \sim p_{\text{simple}}.$$

- **learning** means finding the vector field $u_\theta$ (a neural network)
- **generating** means solving the equation numerically from $t=0$ to $t=1$.

## Flow matching

Take a data sample $x_1 \sim p_{\text{data}}$ and corrupt it with noise $z \sim \mathcal{N}(0, I)$ by an amount that depends on time $t \in [0, 1]$:

$$x_t = \alpha_t\, z + \beta_t\, x_1,$$

with $\alpha_0 = 1,\ \beta_0 = 0$ (pure noise) and $\alpha_1 = 0,\ \beta_1 = 1$ (clean data). We use the simplest choice, $\alpha_t = 1 - t$ and $\beta_t = t$.

The network $u_\theta(x_t, t)$ learns to **denoise**: at every noise level, it predicts the direction in which $x_t$ moves along this path,

$$\frac{dx_t}{dt} = \dot\alpha_t\, z + \dot\beta_t\, x_1 = x_1 - z.$$

This is a regression problem with the loss

$$\mathcal{L}(\theta) = \mathbb{E}_{z,\,x_1,\,t}\,\big\| u_\theta(x_t, t) - (x_1 - z) \big\|^2.$$

## &#128221; Example 1: Training a flow matching model

### Trainer object

In [ ]:
import torch
import torch.nn as nn
from course.model import FlowUNet
from course.training import FlowMatchingTrainer

model_config = dict(n_levels=2)
model = FlowUNet(**model_config)

trainer = FlowMatchingTrainer(
    model     = model,
    criterion = nn.MSELoss(),
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3),
)

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
print(f"device: {trainer.device} | parameters: {n_params / 1e6:.2f} M")

### Get raw data

In [ ]:
### Get data
from course.helper import download_file

fname_train = download_file("https://saco.csic.es/public.php/dav/files/tsK7yQGM7BLaWpR")
fname_val   = download_file("https://saco.csic.es/public.php/dav/files/6BfKXP8Gwm6sdeJ")

In [ ]:
import xarray as xr

da_train = xr.open_dataset(fname_train)["SO2_col_mass"]
da_val   = xr.open_dataset(fname_val)["SO2_col_mass"]

### Dataset and data loader

In [ ]:
from torch.utils.data import DataLoader
from course.dataset import SimulationDataset, LogTransform

coarsen = (4, 4)
transform = LogTransform()

# Datasets
train_dataset = SimulationDataset(da_train, transform, coarsen=coarsen)
val_dataset   = SimulationDataset(da_val,   transform, coarsen=coarsen)

# Dataloaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=16, 
    pin_memory=torch.cuda.is_available(),  # faster CPU -> GPU copies
    shuffle=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    pin_memory=torch.cuda.is_available(),  # faster CPU -> GPU copies
    shuffle=False
)

In [ ]:
C, H, W = train_dataset[0].shape

print("train item shape:", C, H, W)
print("train std:", float(train_dataset.data.std()))           # should be close to 1

In [ ]:
batch = next(iter(train_loader))
print("shape:", tuple(batch.shape), "| dtype:", batch.dtype)   # expected: (B, 1, H, W), torch.float32
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

### Training loop

In [ ]:
history = trainer.fit(train_loader, val_loader, epochs=200)

In [ ]:
history.plot(
    x="epoch",
    y=["train_loss", "val_loss"],
)

In [ ]:
checkpoint = {
    "model_state": model.state_dict(),
    "model_config": model_config,
    "transform": {"factor": 1, "scale": 1},
    "coarsen": coarsen,
    "shape": (H,W),
}
torch.save(checkpoint, "flow_model.pt")

## &#128221; Example 2: Sampling from the trained model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
checkpoint = torch.load("flow_model.pt", map_location=device)

model = FlowUNet(**checkpoint["model_config"])
model.load_state_dict(checkpoint["model_state"])
model = model.to(device).eval()

transform = LogTransform(**checkpoint["transform"])

H, W = checkpoint['shape'] # grid size used in training

In [ ]:
steps = 50
dt = 1.0 / steps
save_at = [0, 5, 10, 20, 30, 40, 50]               # steps at which we store x_t

x = torch.randn(1, 1, H, W, device=device)         # z ~ N(0, I)
frames = {0: x.cpu()}

with torch.no_grad():
    for k in range(steps):
        t = torch.full((1,), k * dt, device=device)
        x = x + dt * model(x, t)
        if k + 1 in save_at:
            frames[k + 1] = x.cpu()

In [ ]:
import matplotlib.pyplot as plt

conf_plt = dict(vmin=0, vmax=5)

fig, axes = plt.subplots(1, len(save_at), figsize=(2.5 * len(save_at), 2.8))
for ax, k in zip(axes, save_at):
    ax.imshow(frames[k][0, 0], **conf_plt)
    ax.set_title(f"t = {k / steps:.2f}")
    ax.axis("off")

In [ ]:
nsamples = 6
steps = 50
dt = 1.0 / steps

x = torch.randn(nsamples, 1, H, W, device=device)         # z ~ N(0, I), (nsamples, C, H, W)

with torch.no_grad():
    for k in range(steps):
        t = torch.full((nsamples,), k * dt, device=device)    # current time, (nsamples,)
        x = x + dt * model(x, t)                              # Euler step

samples = x.cpu()                           

In [ ]:
fig, axes = plt.subplots(1, nsamples, figsize=(3 * nsamples, 3))
for ax, img in zip(axes, samples):
    ax.imshow(img[0], **conf_plt)
    ax.axis("off")
fig.suptitle("Generated")

In [ ]:
real = val_dataset.data[:nsamples]                            # (nsamples, 1, H, W), transformed space

fig, axes = plt.subplots(1, nsamples, figsize=(3 * nsamples, 3))
for ax, img in zip(axes, real):
    ax.imshow(img[0], **conf_plt)
    ax.axis("off")
fig.suptitle("Validation")

In [ ]:
from course.sampling import sample

n = 512
gen = sample(model, (n, 1, H, W), steps=50).cpu()           # transformed space
real = train_dataset.data

In [ ]:
import numpy as np

pts = [(19,19), (12,19), (19,24)]   # (row, col)
n_hist = len(pts)
bins = np.linspace(0, 5, 50)

fig, axes = plt.subplots(1, n_hist, figsize=(4 * n_hist, 3), squeeze=False)
for ax, (i, j) in zip(axes.flat, pts):
    ax.hist(real[:, 0, i, j], bins=bins, density=True, alpha=0.4, label="real")
    ax.hist(gen[:, 0, i, j],  bins=bins, density=True, histtype="step", color="k", linewidth=2, label="generated")
    ax.set_title(f"pixel ({i}, {j})")
plt.legend()
plt.tight_layout()